# Concept detection after debiasing

Computes CAV (LR and Diff-Means) per layer, removes the concept direction via
orthogonal projection, then re-trains classifiers to measure how much concept
information remains.

**Change only `CONCEPT`** — everything else follows automatically.

Outputs:
- `notebooks/results/single_debias/{CONCEPT}/debiased_detection/` — per-layer CSVs + plot
- `data/single_debiasing/{CONCEPT}/cavs/layer_XX.npz` — CAV vectors (keys: `lr_cav`, `dm_cav`)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from xgboost import XGBClassifier
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'pyproject.toml').exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from software.metrics import get_metrics
from software.torch_lr import TorchLR
from software.viz import plot_debiased_detection

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
CONCEPT        = 'eyeglasses'
NUM_LAYERS     = 24
RUNS_PER_LAYER = 10

# Solver for logistic regression (both CAV training and probing):
#   'torch_lr' — TorchLR (LBFGS on GPU, stable, recommended)
#   'sgd'      — SGDClassifier from sklearn (stochastic gradient descent)
SOLVER   = 'torch_lr'
SOLVER_C = 0.1

METADATA_PATH   = ROOT / 'data' / 'metadata.csv'
ACTIVATIONS_DIR = ROOT / 'data' / 'activations' / 'raw'
CAV_OUT_DIR     = ROOT / 'data' / 'activations' / 'debiased' / 'single' / CONCEPT / 'cavs_analysis'
OUT_DIR = ROOT / 'notebooks' / 'results' / 'single_debias' / CONCEPT / 'debiased_detection'
CAV_OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

meta = pd.read_csv(METADATA_PATH)[['filename', 'split', CONCEPT]]
meta_train = meta[meta['split'] == 'train'][['filename', CONCEPT]]
meta_test  = meta[meta['split'] == 'test'][['filename',  CONCEPT]]

print(f'Concept : {CONCEPT}')
print(f'Solver  : {SOLVER} (C={SOLVER_C})')
print(f'Output  : {OUT_DIR}')
print(f'CAVs    : {CAV_OUT_DIR}')

In [ ]:
def make_clf(random_state=None):
    if SOLVER == 'torch_lr':
        return TorchLR(C=SOLVER_C, max_iter=500, random_state=random_state)
    return SGDClassifier(
        loss='log_loss', penalty='l2',
        alpha=1.0 / (2.0 * SOLVER_C),
        max_iter=1000, tol=1e-4,
        random_state=random_state,
    )


def load_layer(split_label, layer_idx):
    path = ACTIVATIONS_DIR / split_label / f'layer_{layer_idx:02d}.parquet'
    assert path.exists(), f'Missing: {path}. Run 02_get_activations.ipynb first.'
    df = pd.read_parquet(path)
    meta_split = meta_train if split_label == 'train' else meta_test
    df = df.merge(meta_split, on='filename')
    feat_cols = [c for c in df.columns if c not in ('filename', CONCEPT)]
    return df[feat_cols].values.astype(np.float32), df[CONCEPT].astype(int).values


def get_debiased_datasets(X_train, y_train, X_test):
    """Trains LR-CAV and DM-CAV, returns orthogonally projected train/test pairs."""
    # LR CAV
    lr_finder = make_clf(random_state=42)
    lr_finder.fit(X_train, y_train)
    cav_lr = lr_finder.coef_[0]
    cav_lr_norm = cav_lr / (np.linalg.norm(cav_lr) + 1e-10)

    # DM CAV
    cav_dm = X_train[y_train == 1].mean(axis=0) - X_train[y_train == 0].mean(axis=0)
    cav_dm_norm = cav_dm / (np.linalg.norm(cav_dm) + 1e-10)

    def project_out(X, v):
        return X - np.dot(X, v).reshape(-1, 1) * v

    return (
        {'lr': (project_out(X_train, cav_lr_norm), project_out(X_test, cav_lr_norm)),
         'dm': (project_out(X_train, cav_dm_norm), project_out(X_test, cav_dm_norm))},
        cav_lr_norm.astype(np.float32),
        cav_dm_norm.astype(np.float32),
    )

In [ ]:
for layer in tqdm(range(NUM_LAYERS), desc='Layers'):
    X_train_raw, y_train_raw = load_layer('train', layer)
    X_test_raw,  y_test_raw  = load_layer('test',  layer)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_test_scaled  = scaler.transform(X_test_raw)

    layer_results = []
    cav_lr_accum = np.zeros(X_train_raw.shape[1], dtype=np.float32)
    cav_dm_accum = np.zeros(X_train_raw.shape[1], dtype=np.float32)

    for run_id in tqdm(range(RUNS_PER_LAYER), desc=f'L{layer:02d}', leave=False):
        X_b, y_b = resample(
            X_train_scaled, y_train_raw,
            replace=True, n_samples=len(y_train_raw),
            random_state=run_id, stratify=y_train_raw,
        )

        debiased, cav_lr, cav_dm = get_debiased_datasets(X_b, y_b, X_test_scaled)
        cav_lr_accum += cav_lr
        cav_dm_accum += cav_dm

        for method, (X_tr_clean, X_te_clean) in debiased.items():
            probe_models = {
                'LogisticRegression': make_clf(random_state=run_id),
                'XGBoost': XGBClassifier(
                    n_estimators=100, max_depth=4,
                    tree_method='hist', device='cuda',
                    random_state=run_id,
                ),
            }
            for model_name, model in probe_models.items():
                model.fit(X_tr_clean, y_b)
                tr_m = get_metrics(y_b,        model.predict(X_tr_clean), model.predict_proba(X_tr_clean)[:, 1])
                te_m = get_metrics(y_test_raw,  model.predict(X_te_clean), model.predict_proba(X_te_clean)[:, 1])
                layer_results.append({
                    'layer_id': layer, 'run_id': run_id,
                    'debias_method': method, 'model': model_name,
                    **{f'train_{k}': v for k, v in tr_m.items()},
                    **{f'test_{k}':  v for k, v in te_m.items()},
                })

    pd.DataFrame(layer_results).to_csv(OUT_DIR / f'{layer:02d}_debiased_results.csv', index=False)
    np.savez(
        CAV_OUT_DIR / f'layer_{layer:02d}.npz',
        lr_cav=(cav_lr_accum / RUNS_PER_LAYER).astype(np.float32),
        dm_cav=(cav_dm_accum / RUNS_PER_LAYER).astype(np.float32),
    )

## Visualization

In [ ]:
import glob

all_files = sorted(glob.glob(str(OUT_DIR / '*_debiased_results.csv')))
df_all = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True).sort_values('layer_id')

metric_cols = [
    c for c in df_all.columns
    if ('train_' in c or 'test_' in c) and c.split('_')[1] not in ('recall', 'precision')
]
id_vars = ['layer_id', 'model', 'run_id', 'debias_method']

df_melted = df_all.melt(id_vars=id_vars, value_vars=metric_cols,
                        var_name='metric_type', value_name='score')
df_melted['subset'] = df_melted['metric_type'].str.split('_').str[0]
df_melted['metric'] = df_melted['metric_type'].str.split('_').str[1]

plot_debiased_detection(
    df_melted, CONCEPT,
    save_path=OUT_DIR / f'debiased_detection_{CONCEPT}.png',
)